In [1]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from celery.utils.time import is_naive
from gitdb.util import rename


# Air Pollution and Health Impact Analysis

## 1. Introduction
Air pollution is a significant global issue that affects human health.
Pollutants such as PM2.5, NO₂, and CO₂ are associated with respiratory and cardiovascular diseases.

In this project, we explore how air pollution levels relate to health outcomes across countries.

---

## 2. Goal of the Analysis
The goal of this project is to:

- Combine air pollution and health datasets
- Explore relationships between pollution and health indicators
- Quantify these relationships using statistical methods
- Build simple models to predict health outcomes

---

## 3. Data Sources
We will use two independent datasets:

- Air pollution dataset: *Our World in Data (https://ourworldindata.org/grapher/pm25-air-pollution)*
- Health dataset: *World Bank Group (https://data.worldbank.org/indicator/SP.DYN.LE00.IN)*

---

## 4. Methodology Overview
The analysis will follow these steps:

1. Data loading
2. Data cleaning
3. Data merging
4. Exploratory Data Analysis (EDA)
5. Statistical analysis (correlation)
6. Predictive modeling

---

## 5. Notes
- Correlation does not imply causation
- Data from different sources may have inconsistencies

---

## 1. Data Loading

In this section, we load the datasets used in the analysis:
- Air pollution dataset (PM2.5 levels)
- Health dataset (life expectancy)

In [2]:
air_df = pd.read_csv('../data/raw/pm25-air-pollution.csv')
health_df = pd.read_excel('../data/raw/API_SP.DYN.LE00.IN_DS2_en_excel_v2_207158.xls')

## 2. Initial Data Inspection

We explore the structure of both datasets to understand:
- available columns
- data types
- potential issues

In [3]:
# Air Polution Dataset (first few rows)
air_df.head()


,Entity,Code,Year,Concentrations of fine particulate matter (PM2.5) - Residence area type: Total
0,Afghanistan,AFG,2010,68.96605
1,Afghanistan,AFG,2011,66.94454
2,Afghanistan,AFG,2012,68.25744
3,Afghanistan,AFG,2013,72.18289
4,Afghanistan,AFG,2014,68.05715


In [4]:
air_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2100 entries, 0 to 2099
Data columns (total 4 columns):
 #   Column                                                                          Non-Null Count  Dtype  
---  ------                                                                          --------------  -----  
 0   Entity                                                                          2100 non-null   str    
 1   Code                                                                            1960 non-null   str    
 2   Year                                                                            2100 non-null   int64  
 3   Concentrations of fine particulate matter (PM2.5) - Residence area type: Total  2100 non-null   float64
dtypes: float64(1), int64(1), str(2)
memory usage: 65.8 KB


In [5]:
health_df.head()

,Data Source,World Development Indicators,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 60,Unnamed: 61,Unnamed: 62,Unnamed: 63,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67,Unnamed: 68,Unnamed: 69
0,Last Updated Date,2026-02-24 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Country Name,Country Code,Indicator Name,Indicator Code,1960.000000,1961.000000,1962.00000,1963.000000,1964.000000,1965.000000,...,2016.000000,2017.000000,2018.000000,2019.000000,2020.000000,2021.000000,2022.00000,2023.000000,2024.0,2025.0
3,Aruba,ABW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,64.049000,64.215000,64.60200,64.944000,65.303000,65.615000,...,75.540000,75.620000,75.880000,76.019000,75.406000,73.655000,76.22600,76.353000,NaN,NaN
4,Africa Eastern and Southern,AFE,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,44.169658,44.468838,44.87789,45.160583,45.535695,45.770723,...,62.167981,62.591275,63.330691,63.857261,63.766484,62.979999,64.48702,65.146154,NaN,NaN


In [6]:
health_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 269 entries, 0 to 268
Data columns (total 70 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Data Source                   268 non-null    str    
 1   World Development Indicators  268 non-null    object 
 2   Unnamed: 2                    267 non-null    str    
 3   Unnamed: 3                    267 non-null    str    
 4   Unnamed: 4                    264 non-null    float64
 5   Unnamed: 5                    265 non-null    float64
 6   Unnamed: 6                    265 non-null    float64
 7   Unnamed: 7                    264 non-null    float64
 8   Unnamed: 8                    264 non-null    float64
 9   Unnamed: 9                    264 non-null    float64
 10  Unnamed: 10                   265 non-null    float64
 11  Unnamed: 11                   265 non-null    float64
 12  Unnamed: 12                   265 non-null    float64
 13  Unnamed: 13     

## 3. Cleaning the Health Dataset

Initial inspection shows that:
- the first 3 rows contain metadata
- actual column names start from row 4

Therefore, we reload the dataset while skipping these rows.

In [7]:
health_df = pd.read_excel(
    '../data/raw/API_SP.DYN.LE00.IN_DS2_en_excel_v2_207158.xls',
    skiprows=3,
)

health_df.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,64.049000,64.215000,64.602000,64.944000,65.303000,65.615000,...,75.540000,75.620000,75.880000,76.019000,75.406000,73.655000,76.226000,76.353000,NaN,NaN
1,Africa Eastern and Southern,AFE,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,44.169658,44.468838,44.877890,45.160583,45.535695,45.770723,...,62.167981,62.591275,63.330691,63.857261,63.766484,62.979999,64.487020,65.146154,NaN,NaN
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,32.799000,33.291000,33.757000,34.201000,34.673000,35.124000,...,62.646000,62.406000,62.443000,62.941000,61.454000,60.417000,65.617000,66.035000,NaN,NaN
3,Africa Western and Central,AFW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,37.779636,38.058956,38.681792,38.936918,39.194580,39.479784,...,56.392452,56.626439,57.036976,57.149847,57.364425,57.362572,57.987813,58.855722,NaN,NaN
4,Angola,AGO,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,37.933000,36.902000,37.168000,37.419000,37.704000,37.968000,...,61.619000,62.122000,62.622000,63.051000,63.116000,62.958000,64.246000,64.617000,NaN,NaN


In [8]:
health_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 70 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    str    
 1   Country Code    266 non-null    str    
 2   Indicator Name  266 non-null    str    
 3   Indicator Code  266 non-null    str    
 4   1960            263 non-null    float64
 5   1961            264 non-null    float64
 6   1962            264 non-null    float64
 7   1963            263 non-null    float64
 8   1964            263 non-null    float64
 9   1965            263 non-null    float64
 10  1966            264 non-null    float64
 11  1967            264 non-null    float64
 12  1968            264 non-null    float64
 13  1969            264 non-null    float64
 14  1970            264 non-null    float64
 15  1971            264 non-null    float64
 16  1972            264 non-null    float64
 17  1973            264 non-null    float64
 18  1

### 3.1 Removing Unnecessary Columns

The dataset includes:
- indicator metadata columns
- empty year columns (2024, 2025)

These are removed as they are not relevant for the analysis.

In [9]:
health_df = health_df.drop(columns=['Indicator Name', 'Indicator Code', '2024', '2025'])

### 3.2 Reshaping the Dataset

The dataset is in wide format (years as columns).

To enable merging with the air pollution dataset, we convert it into long format using `melt()`.

In [10]:
health_long  = health_df.melt(
    id_vars=['Country Name', 'Country Code'],
    var_name='Year',
    value_name='Life expectancy at birth (total years)',
)

health_long

,Country Name,Country Code,Year,Life expectancy at birth (total years)
0,Aruba,ABW,1960,64.049000
1,Africa Eastern and Southern,AFE,1960,44.169658
2,Afghanistan,AFG,1960,32.799000
3,Africa Western and Central,AFW,1960,37.779636
4,Angola,AGO,1960,37.933000
...,...,...,...,...
17019,Kosovo,XKX,2023,78.033000
17020,"Yemen, Rep.",YEM,2023,69.295000
17021,South Africa,ZAF,2023,66.139000
17022,Zambia,ZMB,2023,66.349000


### 3.3 Data Type Adjustment

The "Year" column is converted to integer for consistency.

In [11]:
health_long.Year = health_long.Year.astype(int)

### 3.4 Missing Values Analysis

We analyze missing values in the Life Expectancy column.

Missing values are not removed at this stage, as they may reflect:
- incomplete reporting
- structural gaps in data

Instead, handling of missing values will be done after aligning datasets.

In [12]:
health_long['Life expectancy at birth (total years)'].isna().value_counts()

Life expectancy at birth (total years)
False    16926
True        98
Name: count, dtype: int64

In [13]:
health_long[health_long['Life expectancy at birth (total years)'].isna()]


,Country Name,Country Code,Year,Life expectancy at birth (total years)
110,Not classified,INX,1960,NaN
115,Israel,ISR,1960,NaN
196,West Bank and Gaza,PSE,1960,NaN
376,Not classified,INX,1961,NaN
462,West Bank and Gaza,PSE,1961,NaN
...,...,...,...,...
15804,Not classified,INX,2019,NaN
16070,Not classified,INX,2020,NaN
16336,Not classified,INX,2021,NaN
16602,Not classified,INX,2022,NaN


## 4. Cleaning the Air Pollution Dataset

The air pollution dataset is already in a relatively tidy format.
However, we still need to:
- rename columns for clarity
- inspect missing values
- keep only the variables relevant for the analysis

In [14]:
air_df.head()

,Entity,Code,Year,Concentrations of fine particulate matter (PM2.5) - Residence area type: Total
0,Afghanistan,AFG,2010,68.96605
1,Afghanistan,AFG,2011,66.94454
2,Afghanistan,AFG,2012,68.25744
3,Afghanistan,AFG,2013,72.18289
4,Afghanistan,AFG,2014,68.05715


In [15]:
air_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2100 entries, 0 to 2099
Data columns (total 4 columns):
 #   Column                                                                          Non-Null Count  Dtype  
---  ------                                                                          --------------  -----  
 0   Entity                                                                          2100 non-null   str    
 1   Code                                                                            1960 non-null   str    
 2   Year                                                                            2100 non-null   int64  
 3   Concentrations of fine particulate matter (PM2.5) - Residence area type: Total  2100 non-null   float64
dtypes: float64(1), int64(1), str(2)
memory usage: 65.8 KB


In [16]:
air_df = air_df.rename(columns={
    'Entity': "Country Name",
    'Code': "Country Code",
    'Concentrations of fine particulate matter (PM2.5) - Residence area type: Total': 'PM2.5',
})

air_df.head()

,Country Name,Country Code,Year,PM2.5
0,Afghanistan,AFG,2010,68.96605
1,Afghanistan,AFG,2011,66.94454
2,Afghanistan,AFG,2012,68.25744
3,Afghanistan,AFG,2013,72.18289
4,Afghanistan,AFG,2014,68.05715


In [17]:
air_df.isna().sum()

Country Name      0
Country Code    140
Year              0
PM2.5             0
dtype: int64

In [18]:
air_df[air_df["Country Code"].isna()]['Country Name'].unique()

<StringArray>
[                                 'Africa (WHO)',
                                'Americas (WHO)',
                'Australia and New Zealand (UN)',
                'Central and Southern Asia (UN)',
      'Eastern Asia and South-Eastern Asia (UN)',
                   'Eastern Mediterranean (WHO)',
                                  'Europe (WHO)',
          'Latin America and the Caribbean (UN)',
         'Northern Africa and Western Asia (UN)',
              'Northern America and Europe (UN)',
 'Oceania (exc. Australia and New Zealand) (UN)',
                         'South-East Asia (WHO)',
                       'Sub-Saharan Africa (UN)',
                         'Western Pacific (WHO)']
Length: 14, dtype: str

### 4.1 Handling Missing Country Codes

Inspection of rows with missing country codes revealed that they represent aggregated regions
(e.g., "Europe (WHO)", "Africa (WHO)", "Latin America and the Caribbean (UN)").

Since this analysis focuses on country-level relationships between air pollution and health outcomes,
these aggregated regions are not suitable and are therefore removed.

In [19]:
air_df = air_df.dropna(subset=['Country Code'])
air_df.isna().sum()

Country Name    0
Country Code    0
Year            0
PM2.5           0
dtype: int64

### 4.2 Year Range Inspection

Before merging datasets, we examine the time coverage of both datasets.

This ensures that comparisons are made only across overlapping years,
avoiding inconsistencies due to missing data in different time periods.

In [20]:
air_df.Year.min(), air_df.Year.max()

(np.int64(2010), np.int64(2019))

In [21]:
health_long.Year.min(), health_long.Year.max()

(np.int64(1960), np.int64(2023))

In [22]:
print(sorted(set(air_df['Year']).intersection(set(health_long['Year']))))

[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]


### 4.3 Time Range Alignment

The air pollution dataset covers the period 2010–2019, while the health dataset spans a much longer timeframe (1960–2023).

To ensure consistency, the analysis is restricted to the overlapping period (2010–2019). This ensures that comparisons between datasets are valid and based on the same temporal context.

In [23]:
common_years = sorted(set(air_df['Year']).intersection(set(health_long['Year'])))

air_df = air_df[air_df['Year'].isin(common_years)]
health_long = health_long[health_long['Year'].isin(common_years)]

In [24]:
air_df.Year.min(), air_df.Year.max()

(np.int64(2010), np.int64(2019))

In [25]:
health_long.Year.min(), health_long.Year.max()

(np.int64(2010), np.int64(2019))

### 4.4 Country Alignment

To ensure consistency, we identify the set of countries present in both datasets.

Only countries that appear in both datasets are retained for further analysis.

In [26]:
# Clean whitespace in Country Name
air_df['Country Name'] = air_df['Country Name'].str.strip()
health_df['Country Code'] = health_df['Country Code'].str.strip()

# Clen whitespace in Country Code
air_df['Country Code'] = air_df['Country Code'].str.strip()
health_df['Country Name'] = health_df['Country Name'].str.strip()

In [27]:
air_countries = set(air_df['Country Name'])
health_countries = set(health_df['Country Name'])

common_countries = air_countries.intersection(health_countries)

print('Air Countries:', len(air_countries))
print('Health Countries:', len(health_countries))
print('Common Countries:', len(common_countries))



Air Countries: 196
Health Countries: 266
Common Countries: 168


In [28]:
air_df = air_df[air_df['Country Name'].isin(common_countries)]
health_long = health_long[health_long['Country Name'].isin(common_countries)]

print('Air Dataset:', len(set(air_df['Country Name'])))
print('Health Dataset:', len(set(health_long['Country Name'])))

Air Dataset: 168
Health Dataset: 168


## 5. Merging Datasets

After aligning both datasets by year and country, we merge them into a single dataset.

This allows us to analyze the relationship between air pollution and health outcomes.

In [29]:
merged_df = pd.merge(
    air_df,
    health_long,
    on=['Country Name', 'Country Code', 'Year'],
    how='inner',
)

merged_df.head()

,Country Name,Country Code,Year,PM2.5,Life expectancy at birth (total years)
0,Afghanistan,AFG,2010,68.96605,60.702
1,Afghanistan,AFG,2011,66.94454,61.250
2,Afghanistan,AFG,2012,68.25744,61.735
3,Afghanistan,AFG,2013,72.18289,62.188
4,Afghanistan,AFG,2014,68.05715,62.260


In [30]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1670 entries, 0 to 1669
Data columns (total 5 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Country Name                            1670 non-null   str    
 1   Country Code                            1670 non-null   str    
 2   Year                                    1670 non-null   int64  
 3   PM2.5                                   1670 non-null   float64
 4   Life expectancy at birth (total years)  1670 non-null   float64
dtypes: float64(2), int64(1), str(2)
memory usage: 65.4 KB


## 5.1 Final Merged Dataset

After cleaning, aligning, and merging both datasets, we obtain a final dataset containing:

- Country Name
- Country Code
- Year
- PM2.5 concentration
- Life expectancy

Each row represents a country-year observation, allowing direct comparison between air pollution and health outcomes.